# `extract_policy_category` — string method নোট

প্রথমে একটা ছোট কিন্তু গুরুত্বপূর্ণ পার্থক্য পরিষ্কার করে নেই, কারণ এটা না জানলে পরে confusion হয়।

**Built-in function বনাম method**

In [ ]:
text = "  Policy Category: Work From Home  \n"

print(len(text))       # built-in function — নাম আগে, জিনিসটা ভেতরে
print(repr(text.strip()))  # method — জিনিসটা আগে, তারপর dot, তারপর নাম

Built-in function গুলো Python নিজেই সবসময় হাতের কাছে রাখে (`len`, `print`, `open`, `sorted`) — কোনো import লাগে না, যেকোনো জায়গায় লেখা যায়। আর method হলো কোনো একটা নির্দিষ্ট type-এর সাথে বাঁধা — `.strip()` শুধু string-এর আছে, list-এর নেই।

এখন মজার ব্যাপার হলো — তোমার `extract_policy_category` function-এ আসলে **একটাও built-in function নেই**। যা যা আছে সবই string method। মানুষ কথার কথায় "built-in" বলে ফেলে, কিন্তু technically এগুলো `str` class-এর method। এটা মাথায় রেখে এগোই।

---

## String method গুলো — একটা একটা করে

### `.splitlines()`

String-কে line ধরে কেটে list বানায়, আর newline character গুলো ফেলে দেয়।

তোমার হয়তো মনে হবে `split("\n")` দিয়েও তো একই কাজ হয়। হয় না — এবং পার্থক্যটা তোমার project-এ আসলেই কাজে লাগবে:

| Input | `.splitlines()` | `.split("\n")` |
|---|---|---|
| `"A\nB"` | `['A', 'B']` | `['A', 'B']` |
| `"A\n"` | `['A']` | `['A', '']` ← বাড়তি ফাঁকা |
| `"A\r\nB"` (Windows) | `['A', 'B']` | `['A\r', 'B']` ← `\r` রয়ে গেল |
| `""` | `[]` | `['']` |

তৃতীয় row-টা আসল কারণ। কেউ যদি Windows-এ policy file লিখে upload করে, file-এ line শেষ হবে `\r\n` দিয়ে। `split("\n")` ব্যবহার করলে তোমার line হতো `'Policy Category: Leave\r'` — শেষে একটা অদৃশ্য carriage return। `.splitlines()` সেটা নিজেই সামলায়। GCS-এ বিভিন্ন লোকের লেখা file আসবে, তাই এই পছন্দটা এমনি এমনি নয়।

In [ ]:
print("A\n\nB".splitlines())   # ['A', '', 'B']

print(f"{'input':<12} {'splitlines()':<22} {'split(newline)'}")
for value in ["A\nB", "A\n", "A\r\nB", ""]:
    print(f"{value!r:<12} {value.splitlines()!r:<22} {value.split('\n')!r}")

### `.strip()`

শুরু আর শেষের whitespace কাটে — space, tab (`\t`), newline (`\n`), carriage return (`\r`)। **মাঝখানের কিছু ছোঁয় না।**

দুইটা আত্মীয় আছে: `.lstrip()` শুধু বাম দিক, `.rstrip()` শুধু ডান দিক।

এখন একটা **ফাঁদ** যেটায় প্রায় সবাই একবার পড়ে। `.strip()`-কে argument দেওয়া যায়, কিন্তু সেটা substring নয় — সেটা **character-এর সেট**:

```python
"textfile.txt".strip(".txt")   # 'extfile'  — ভুল ধারণা হলে অবাক হবে
```

Python এখানে `.txt` শব্দটা খোঁজেনি। সে `{'.', 't', 'x'}` — এই তিনটা character-এর যেকোনোটা দুই প্রান্ত থেকে কাটতে থেকেছে। শুরুর `t` উড়ে গেছে। Extension কাটতে চাইলে `.removesuffix(".txt")` ব্যবহার করবে (Python 3.9+)।

তোমার code-এ argument ছাড়া খালি `.strip()` ব্যবহার হয়েছে, তাই এই সমস্যা নেই — কিন্তু জেনে রাখা ভালো।

In [ ]:
print(repr("   Policy Category: Work From Home  \n".strip()))
# 'Policy Category: Work From Home'

print(repr("textfile.txt".strip(".txt")))   # 'extfile' — শব্দ নয়, character-এর সেট
print("textfile.txt".removesuffix(".txt"))  # 'textfile'

### `.lower()`

সব অক্ষর ছোট হাতের করে **নতুন একটা string** ফেরত দেয়। মূল string বদলায় না, কারণ Python-এ string **immutable** — একবার তৈরি হলে আর পাল্টানো যায় না, শুধু নতুন বানানো যায়।

এই জন্যই তোমার code-এ `stripped = line.strip().lower()` আলাদা variable-এ রাখতে হয়েছে। আর `line` অক্ষত থাকায় শেষে original casing সহ `'Work From Home'` ফেরত দেওয়া সম্ভব হয়েছে।

বাংলা লেখায় `.lower()`-এর কোনো প্রভাব নেই, কারণ বাংলায় case নেই — `"নীতি".lower()` সেই `"নীতি"`-ই থাকে। নিরাপদ, কিছু ভাঙে না।

In [1]:
s = "HELLO"
print(s.lower())   # 'hello'
print(s)           # 'HELLO'  ← s নিজে অপরিবর্তিত

line = "Policy Category: Work From Home"
stripped = line.strip().lower()
print(stripped)
print(line)        # original casing অক্ষত

print("নীতি".lower())

hello
HELLO
policy category: work from home
Policy Category: Work From Home
নীতি


### `.startswith()`

`True`/`False` ফেরত দেয় — string-টা ঠিক ওই অক্ষরগুলো দিয়ে **শুরু** হয়েছে কি না। `in` এর সাথে গুলিয়ে ফেলো না।

Policy document-এর মাঝখানে কোনো বাক্যে কথাটা এসে পড়লে যেন ভুল match না হয় — এই কারণেই `in` নয়, `startswith` ব্যবহার হয়েছে।

একটা কাজের ব্যাপার: এটা **tuple নিতে পারে**, মানে একসাথে কয়েকটা possibility মিলিয়ে দেখতে পারে।

তোমার দুইটা `if`-এর ভেতরের কাজ যেহেতু হুবহু এক, চাইলে এভাবে একটাতে মিলিয়ে দেওয়া যায়। তবে আলাদা রাখার একটা যুক্তিও আছে — কোনটা HR doc আর কোনটা noise doc, সেটা পরে আলাদা করে handle করতে চাইলে দুইটা branch হাতে থাকা ভালো।

আত্মীয়: `.endswith()` — এটা তোমার `document_loader.py`-তেই আছে, `blob.name.endswith(".txt")` লাইনে।

In [ ]:
print("See the policy category: below".startswith("policy category:"))  # False
print("policy category:" in "See the policy category: below")           # True

line = "Policy Category: Work From Home"
stripped = line.strip().lower()
if stripped.startswith(("policy category:", "category:")):
    print(line.split(":", 1)[1].strip())

print("work_from_home_policy.txt".endswith(".txt"))

### `.split(sep, maxsplit)`

দুইটা argument:

- `":"` — কোন character ধরে কাটবে (separator)
- `1` — সর্বোচ্চ কয়বার কাটবে (**maxsplit**)

maxsplit `1` দেওয়ার কারণটা এই তুলনাতেই পরিষ্কার। ধরো ভবিষ্যতে কেউ লিখল `Policy Category: Travel: International`। maxsplit ছাড়া list হতো `['Policy Category', ' Travel', ' International']`, আর `[1]` দিত শুধু `'Travel'` — বাকিটা হারিয়ে যেত। maxsplit `1` দিলে `[1]` পায় `' Travel: International'`, পুরোটাই অক্ষত।

একটা কথা মনে রাখো: separator না পাওয়া গেলে split কোনো error দেয় না, পুরো string-কে এক element-এর list বানায়। তাহলে `[1]` করলে **IndexError** হতো। তোমার code-এ এটা কখনো ঘটবে না, কারণ `startswith(...":")` আগেই নিশ্চিত করে ফেলেছে colon আছে। এই দুইটা line একসাথে কাজ করে — guard আর extraction।

In [ ]:
print("Policy Category: Work From Home".split(":", 1))
# ['Policy Category', ' Work From Home']

print("a:b:c".split(":"))       # ['a', 'b', 'c']  ← maxsplit ছাড়া, সব কাটে
print("a:b:c".split(":", 1))    # ['a', 'b:c']     ← একবারই কাটে

print("Policy Category: Travel: International".split(":")[1])
print("Policy Category: Travel: International".split(":", 1)[1].strip())

print("no colon here".split(":", 1))   # ['no colon here'] — [1] করলে IndexError

### `.rsplit()` — (মূল file-এ আছে)

`split` কাটে বাম থেকে, `rsplit` কাটে **ডান থেকে**। GCS blob name হয় `hr_policies/raw/work_from_home_policy.txt` — শুধু শেষের filename চাই।

আর `[-1]` মানে list-এর শেষ element। Negative index Python-এ পেছন থেকে গোনে: `-1` শেষটা, `-2` তার আগেরটা।

In [ ]:
filename = "hr_policies/raw/work_from_home_policy.txt".rsplit("/", 1)[-1]
print(filename)

print("hr/raw/wfh.txt".split("/", 1))    # ['hr', 'raw/wfh.txt']  ← বাম থেকে
print("hr/raw/wfh.txt".rsplit("/", 1))   # ['hr/raw', 'wfh.txt']  ← ডান থেকে
print("hr/raw/wfh.txt".rsplit("/", 1)[-1])

### `.append()` — (মূল file-এ আছে)

```python
documents.append(Document(...))
```

List method — শেষে একটা element জুড়ে দেয়। এটা **in-place** কাজ করে, মানে list-টাকেই বদলে দেয়, আর **`None` return করে**।

এখান থেকেই beginner-দের একটা ক্লাসিক bug আসে: assignment-এর বাঁ পাশে `.append()` রাখলে list হারিয়ে `None` হয়ে যায়।

In [ ]:
documents = []
doc = {"source": "work_from_home_policy.txt"}

documents = documents.append(doc)   # return None, তাই documents এখন None
print(documents)

documents = []
documents.append(doc)               # list-টা নিজে বদলায়
print(documents)

---

## Function নয়, কিন্তু function-এর মতো দেখতে — slicing আর indexing

```python
text.splitlines()[:5]     # slice
line.split(":", 1)[1]     # index
```

এগুলোর পরে বন্ধনী নেই, কারণ এগুলো method নয় — এগুলো Python-এর **syntax**, যা ভেতরে ভেতরে `__getitem__` নামের জিনিস ডাকে।

Slice-এর নিয়ম `[start:stop]`, যেখানে **stop বাদ যায়**।

Slice-এর একটা দারুণ নিরাপত্তা আছে — **range পার হয়ে গেলেও error দেয় না**।

File-এ যদি মাত্র ২ line থাকে, `[:5]` চুপচাপ ২টাই দেয়। অথচ index দিয়ে `lines[4]` লিখলে IndexError হতো। ছোট file নিয়ে দুশ্চিন্তা করতে হয় না এই কারণে।

In [ ]:
lines = ["a", "b", "c", "d", "e", "f", "g"]
print(lines[:5])     # index 0 থেকে 4 — stop বাদ
print(lines[2:5])
print(lines[-2:])    # শেষ দুইটা

print("one line only".splitlines()[:5])   # ['one line only'] — crash নয়

---

## `json.loads()` — built-in নয়, standard library

```python
record = json.loads(blob.download_as_text())
```

এটা `import json` করতে হয়, তাই built-in নয় — standard library-র অংশ। JSON-format-এ লেখা **string** নিয়ে Python-এর dict/list বানায়।

নামের শেষের `s` মানে **string**। এই জোড়াটা গুলিয়ে ফেলা খুব সাধারণ ভুল:

| Function | Input | কখন |
|---|---|---|
| `json.loads(s)` | string | GCS থেকে text নামিয়ে আনার পর — **তোমার case** |
| `json.load(f)` | file object | `open()` দিয়ে খোলা file থেকে |
| `json.dumps(obj)` | → string | GCS-এ লেখার আগে (processor.py-তে) |
| `json.dump(obj, f)` | → file-এ লেখে | local file-এ সরাসরি |

`loads` হয়ে যাওয়ার পর `record` একটা সাধারণ Python dict, তাই `record["text"]` দিয়ে value তোলা যায়।

In [ ]:
import json

raw = '{"text": "Policy Category: Leave"}'
record = json.loads(raw)          # string → dict
print(record["text"])
print(json.dumps(record))         # dict → string

---

## যেগুলো আসলেই built-in function — notebook-এ যা ব্যবহার করেছিলে

| Function | কাজ | কেন কাজে লাগল |
|---|---|---|
| `print(x)` | screen-এ দেখায় | output যাচাই |
| `repr(x)` | debug-উপযোগী রূপ দেখায় | লুকানো space ধরা পড়ে |
| `open(path)` | file খোলে | local test |
| `sorted(it)` | sort করা নতুন list | file গুলো নির্দিষ্ট ক্রমে |
| `len(x)` | দৈর্ঘ্য | কয়টা document এলো |
| `enumerate(it)` | index + value একসাথে | line number সহ ঘোরা |

`print` আর `repr`-এর পার্থক্যটা আবার মনে করিয়ে দেই, কারণ debugging-এ এটা তোমার সবচেয়ে কাজের অস্ত্র।

`.strip()` ঠিকমতো কাজ করছে কি না, এটাই সবচেয়ে দ্রুত যাচাইয়ের উপায়।

In [ ]:
value = " Work From Home "
print(value)         # space দেখা যাচ্ছে না
print(repr(value))   # ' Work From Home '  ← ধরা পড়ল
print(repr(value.strip()))

---

## এক নজরে

পুরো function-টা মাত্র **চারটা string method** (`splitlines`, `strip`, `lower`, `startswith`, `split` — পাঁচটা) আর **দুইটা syntax feature** (slice, index) দিয়ে তৈরি। কোনো regular expression নেই, কোনো external library নেই। এটাই ভালো ingestion code-এর চেহারা — যত কম নির্ভরতা, তত কম জায়গায় ভাঙার সুযোগ।

In [ ]:
def extract_policy_category(text: str) -> str:
    for line in text.splitlines()[:5]:        # splitlines: str method | [:5]: slice syntax
        stripped = line.strip().lower()       # strip + lower: দুইটাই str method
        if stripped.startswith("policy category:"):   # startswith: str method → bool
            return line.split(":", 1)[1].strip()      # split: str method | [1]: index
        if stripped.startswith("category:"):
            return line.split(":", 1)[1].strip()
    return "Unknown"

In [ ]:
sample = """WORK FROM HOME POLICY

Policy Category: Work From Home

1. Purpose
This policy outlines the eligibility..."""

print(repr(extract_policy_category(sample)))
sample.splitlines()[:5]